# Prioritizing Organic Search Content Decay: Research Framing & Problem Formulation

**Dataset Under Contract:** `data/raw/content_refresh_anonymized.csv` (30,000 observations across 32 enterprise clients)  
**System Purpose:** Decision Support, Content Refresh Prioritization, and Editorial ROI Optimization  
**Technical Reference:** `docs/data-dictionary.md` & `notebooks/02_data_contract.ipynb`

---

## Executive Summary & Business Motivation
In modern digital publishing and search engine optimization (SEO), web content follows distinct lifecycle patterns. After publication and initial search engine ranking, many pages inevitably experience **content decay**—a steady, unobserved loss in search rankings, impression share, and organic traffic. For enterprise content teams managing portfolios of 10,000 to over 100,000 URLs across diverse client websites, manually auditing every single page each month is cost-prohibitive.

Organizations often rely on crude heuristic rules—such as static traffic drops or arbitrary age thresholds—to decide which articles need refreshing. However, these simplistic rules frequently fail because search performance is multidimensional and nonlinear: an article's health depends on user engagement, competitive pressure, keyword search volume, and content maturity.

**The Core Business Decision:** *Which underperforming pages should content strategy teams refresh first to maximize organic traffic recovery while minimizing wasted editorial hours?*

---

### The Core Business Framework (Four Key Decision Questions)
To ensure this data science initiative delivers measurable operational value, the research is grounded in four foundational business questions:
1. **What operational decision does this improve?**
   - It transforms the weekly editorial refresh queue from reactive guesswork or noisy static rules into a rank-ordered, risk-weighted intervention pipeline.
2. **Who acts on the recommendations, and what action do they take?**
   - Content strategists, SEO directors, and editorial teams allocate fixed sprint hours by reviewing the top-ranked at-risk URLs, updating outdated statistics, enhancing depth, and optimizing metadata.
3. **What is the commercial cost of decision errors?**
   - **False Positive (Type I Error):** Editors spend valuable billable hours rewriting healthy or temporarily fluctuating pages that would have self-corrected, generating minimal incremental return on investment.
   - **False Negative (Type II Error):** Compounding, undetected traffic erosion on high-value pillar content, allowing competing websites to permanently capture Page 1 search rankings.
4. **Why does machine learning deliver superior ROI over basic heuristic rules?**
   - Search dynamics involve complex interactions between user behavior (CTR, scroll depth, engagement rate), ranking trajectory, keyword economics (CPC, competition), and content format across diverse client domains. Machine learning isolates subtle decay patterns that static rules miss, while filtering out benign fluctuations.

### The Executive Problem Statement
> *For content strategists deciding which articles to refresh, this study investigates whether machine learning models can rank decaying content more accurately than traditional heuristic rules using 90-day search visibility and user engagement signals. Models are evaluated directly on operational decision metrics (Precision@K and out-of-domain ROC-AUC against natural baseline rates). Every recommendation is designed to save editorial hours, defend high-value traffic, and deliver verifiable business impact.*


## 1. Environment Setup & Data Ingestion

We load the anonymized enterprise benchmark dataset `content_refresh_anonymized.csv`. This dataset provides a cross-sectional snapshot of 30,000 content assets across 32 distinct enterprise publisher domains. It integrates search visibility metrics from Google Search Console (GSC) with behavioral engagement metrics from Google Analytics 4 (GA4) over a standard 90-day observation window.


In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
import os
# Robust path resolution
DATA_PATH = "../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"Dataset not found at {DATA_PATH}"

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded raw dataset: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns.")

Loaded raw dataset: 30,000 rows and 44 columns.


## 2. Unit of Analysis & Dataset Grain Verification

In enterprise analytics, a foundational prerequisite is establishing the **unit of analysis (grain)** before exploratory manipulation or modeling:
- **Unit of Analysis:** Exactly one row represents one unique published content asset (`content_id`).
- **Integrity Test:** We programmatically verify that `unique(content_id) == total_rows`, ensuring there are no duplicated records, accidental join artifacts, or multi-period distortions.


In [3]:
total_rows = len(df_raw)
unique_content = df_raw["content_id"].nunique()
duplicate_count = total_rows - unique_content

print(f"Total Rows: {total_rows:,}")
print(f"Unique content_id: {unique_content:,}")
print(f"Duplicate content_id rows: {duplicate_count:,}")

Total Rows: 30,000
Unique content_id: 30,000
Duplicate content_id rows: 0


## 3. Data Integrity & Missingness Pattern Audit

In real-world enterprise data, missing values rarely occur purely at random. Missingness patterns convey important operational and architectural context:
- `provider_used` (71.5% unpopulated) and `model_used` (19.1% unpopulated): Differentiate human-authored articles from AI-assisted content workflows.
- `word_count` / `char_count` (25.7% unpopulated): Distinguish rich editorial articles from non-text assets (such as interactive calculators, landing pages, and media hubs).
- `search_volume`, `cpc`, `competition` (~8.2% unpopulated): Reflect long-tail or unbranded queries where external keyword index data is unavailable.

> **Engineering Best Practice:** Missing values are not arbitrarily zero-filled, as doing so introduces artificial noise. Instead, structural missingness indicators (`has_*`) are preserved to inform downstream modeling.


In [4]:
# Inspect null rates across all fields
missing_summary = df_raw.isnull().mean() * 100
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)


if missing_summary.empty:
    print("No missing values found!")
else:
    print("Columns with missing values:")
    print(missing_summary)

Columns with missing values:
provider_used        71.460000
word_count_tier      25.663333
char_count           25.663333
word_count           25.663333
char_count_tier      25.663333
model_used           19.110000
trend_pct            11.293333
competition_level     8.700000
search_volume         8.226667
competition           8.226667
cpc                   8.226667
main_intent           7.913333
scroll_rate           0.416667
dtype: float64


## 4. Cohort Eligibility & Target Formulation

### 4.1 Business Eligibility Criteria
To ensure an honest, decision-grade evaluation, we focus on mature, active assets where content decay is genuinely observable:
1. **Search Visibility Floor:** `impressions_90d > 0` ensures the URL has active search visibility and is indexed by search engines.
2. **Lifecycle Maturity:** `content_age_days >= 90` ensures the article has progressed past its initial launch volatility window into a stabilized lifecycle phase.

### 4.2 Target Label Formulation & The Natural Base Rate
- **Target Variable (`is_declining_label`):** Binary business outcome indicating significant organic slippage (`trend_direction == 'down'`).
- **The Natural Base Rate:** In any classification problem, establishing the natural base rate is paramount. A naive rule predicting decay for every page would match this rate by chance alone. To prove commercial value, any proposed machine learning system must demonstrate statistically superior precision on the top-ranked queue (Precision@K) and overall discrimination (ROC-AUC) against this benchmark.


In [5]:
# Standard eligibility filtering as defined in repo guidelines:
# 1. Content must have at least some search exposure (impressions_90d > 0)
# 2. Content must be mature enough to assess decay (content_age_days >= 90)
df_eligible = df_raw[(df_raw["impressions_90d"] > 0) & (df_raw["content_age_days"] >= 90)].copy()

# Construct the proxy target label
df_eligible["is_declining_label"] = (df_eligible["trend_direction"] == "down").astype(int)
base_rate = df_eligible["is_declining_label"].mean()
counts = df_eligible["trend_direction"].value_counts()

print(f"Eligible content items: {len(df_eligible):,} / {len(df_raw):,} ({len(df_eligible)/len(df_raw):.1%})")
print(f"Target Base Rate (is_declining_label == 1): {base_rate:.2%}")
print("\nTrend Direction breakdown:")
for trend, cnt in counts.items():
    print(f"  - {trend:8s}: {cnt:6,d} ({cnt/len(df_eligible):.2%})")

Eligible content items: 30,000 / 30,000 (100.0%)
Target Base Rate (is_declining_label == 1): 54.21%

Trend Direction breakdown:
  - down    : 16,262 (54.21%)
  - stable  :  5,962 (19.87%)
  - up      :  4,388 (14.63%)
  - new     :  2,236 (7.45%)
  - flat    :  1,152 (3.84%)


## 5. Research Conclusions & Project Roadmap

### Key Empirical Findings
1. **Dataset Grain Confirmed:** The unit of analysis strictly holds at 1 row per unique `content_id` across all 30,000 observations.
2. **Cohort Eligibility:** 100% of the sample meets the maturity requirements (`content_age_days >= 90` and `impressions_90d > 0`), providing a robust evaluation foundation.
3. **Natural Base Rate:** Downward-trending articles account for **54.21%** (n = 16,262) of the portfolio, establishing the exact benchmark that predictive models must outperform
